# 03. Análisis inicial, limpieza y representaciones del dataset

La lógica de limpieza y transformación vive en el paquete
`wisconsin_cancer_analysis` (`dataset.py` y `features.py`); este notebook
solo orquesta las llamadas y documenta las decisiones.


In [1]:
import sys
from pathlib import Path

# Permite importar el paquete sin instalarlo (equivalente a `pip install -e .`)
sys.path.insert(0, str(Path('..').resolve()))
import pandas as pd

from wisconsin_cancer_analysis.dataset import clean_dataset, fetch_dataset, load_raw
from wisconsin_cancer_analysis.features import (
    REPRESENTATIONS,
    build_rep1_zscore,
    build_rep2_binning,
    build_rep2_kmeans,
    build_rep3_atipia,
    kmeans_umbrales,
    load_clean,
)
from wisconsin_cancer_analysis.config import PROCESSED_DATA_DIR, RAW_DATA_DIR, REPORTS_DIR


2026-11-03 18:53:17.707 | INFO     | wisconsin_cancer_analysis.config:<module>:11 - PROJ_ROOT path is: C:\Users\patyq\OneDrive\Documentos\wisconsin-cancer-analysis


Justificación del Limpiado Previo (Común para ambas propuestas)
- Exclusión de Sample_code_number: Se descarta por ser un identificador administrativo de paciente que no aporta valor informativo ni biológico, previniendo el sobreajuste (overfitting) por memorización de patrones irrelevantes [7, 8].
- Imputación por mediana en Bare_nuclei: Permite conservar la totalidad del conjunto de datos original sin descartar observaciones, empleando la mediana como un estimador estadístico robusto ante sesgos en variables discretas [7, 8].


In [2]:
# Carga (o descarga con fetch_dataset() si no existe el CSV) y limpieza:
#   - elimina Sample_code_number y duplicados
#   - '?' -> NaN e imputacion por mediana en Bare_nuclei
#   - Class: 2 (Benigno) -> 0, 4 (Maligno) -> 1
raw_path = RAW_DATA_DIR / 'breast_cancer_original.csv'
df_clean = clean_dataset(load_raw(raw_path))
df_clean.to_csv(PROCESSED_DATA_DIR / 'breast_cancer_limpio.csv', index=False)
df_clean.head()


2026-11-03 18:53:23.041 | INFO     | wisconsin_cancer_analysis.dataset:load_raw:44 - Datos crudos cargados: (699, 10) desde C:\Users\patyq\OneDrive\Documentos\wisconsin-cancer-analysis\data\raw\breast_cancer_original.csv
2026-11-03 18:53:23.056 | INFO     | wisconsin_cancer_analysis.dataset:clean_dataset:63 - Filas duplicadas eliminadas: 236
2026-11-03 18:53:23.160 | INFO     | wisconsin_cancer_analysis.dataset:clean_dataset:70 - Valores nulos restantes: 0


,Clump_thickness,Uniformity_of_cell_size,Uniformity_of_cell_shape,Marginal_adhesion,Single_epithelial_cell_size,Bare_nuclei,Bland_chromatin,Normal_nucleoli,Mitoses,Class
0,5.0,1.0,1.0,1.0,2.0,1.0,3.0,1.0,1.0,0
1,5.0,4.0,4.0,5.0,7.0,10.0,3.0,2.0,1.0,0
2,3.0,1.0,1.0,1.0,2.0,2.0,3.0,1.0,1.0,0
3,6.0,8.0,8.0,1.0,3.0,4.0,3.0,7.0,1.0,0
4,4.0,1.0,1.0,3.0,2.0,1.0,3.0,1.0,1.0,0


#### Representación 1: Numérica Estandarizada (Z-score)
- Objetivo: Mantener la variable como continua pero llevarla a media 0 y desviación estándar 1.
- Uso: Ideal para Naive Bayes, Regresión Logística o algoritmos basados en distancias como KNN.


In [3]:
df_rep1 = build_rep1_zscore(df_clean)
df_rep1.to_csv(PROCESSED_DATA_DIR / 'representacion_1_zscore.csv', index=False)
df_rep1.head()


,Clump_thickness_zscore,Uniformity_of_cell_size_zscore,Uniformity_of_cell_shape_zscore,Marginal_adhesion_zscore,Single_epithelial_cell_size_zscore,Bare_nuclei_zscore,Bland_chromatin_zscore,Normal_nucleoli_zscore,Mitoses_zscore,Class
0,-0.112858,-0.981089,-1.038913,-0.854637,-0.752852,-0.979626,-0.444067,-0.828985,-0.433845,0
1,-0.112858,-0.054060,-0.076880,0.415658,1.292749,1.370462,-0.444067,-0.533284,-0.433845,0
2,-0.809565,-0.981089,-1.038913,-0.854637,-0.752852,-0.718505,-0.444067,-0.828985,-0.433845,0
3,0.235496,1.181979,1.205832,-0.854637,-0.343732,-0.196264,-0.444067,0.945221,-0.433845,0
4,-0.461211,-0.981089,-1.038913,-0.219489,-0.752852,-0.979626,-0.444067,-0.828985,-0.433845,0


Justificación de la Representación 1: Numérica Estandarizada (Z-score)
- Técnicas aplicadas: Imputación por mediana de la variable Bare_nuclei y estandarización Z-score en las 9 características clínicas (media 0, desviación estándar 1) [7, 8].
- Fundamento en Ingeniería de Características: Mantiene la naturaleza continua y cuantitativa original de las mediciones histológicas sin distorsionar la distribución ni la distancia relativa entre las observaciones [7, 8]. Al igualar la media y la varianza de todas las variables, se evita que aquellos atributos con mayor dispersión dominen injustificadamente sobre los demás durante el aprendizaje [7, 8].
- Idoneidad Algorítmica: Es la representación recomendada para clasificadores probabilísticos como Naive Bayes (al trabajar sobre distribuciones numéricas continuas homogeneizadas) y para modelos basados en distancias como K-Nearest Neighbors (KNN), Máquinas de Vectores de Soporte (SVM) o Regresión Logística [1, 3, 5].


#### Representación 2: Discretización por Rangos de Severidad (Binning)
- Objetivo: Convertir la escala del 1 al 10 en categorías clínicas interpretables.
- Criterio de agrupación:
    - 1 a 3 $\rightarrow$ 0 (Normal / Bajo)
    - 4 a 6 $\rightarrow$ 1 (Moderado)
    - 7 a 10 $\rightarrow$ 2 (Alto / Atípico)
- Uso: Ideal para generar reglas claras de corte en Árboles de Decisión.


In [4]:
df_rep2 = build_rep2_binning(df_clean)
df_rep2.to_csv(PROCESSED_DATA_DIR / 'representacion_2_binning.csv', index=False)
df_rep2.head()


,Clump_thickness_bin,Uniformity_of_cell_size_bin,Uniformity_of_cell_shape_bin,Marginal_adhesion_bin,Single_epithelial_cell_size_bin,Bare_nuclei_bin,Bland_chromatin_bin,Normal_nucleoli_bin,Mitoses_bin,Class
0,1,0,0,0,0,0,0,0,0,0
1,1,1,1,1,2,2,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,1,2,2,0,0,1,0,2,0,0
4,1,0,0,0,0,0,0,0,0,0


Justificación de la Representación 2: Discretización por Rangos Clínicos (Binning)
- Técnicas aplicadas: Imputación por mediana y agrupamiento en 3 categorías ordinales de severidad (0: Bajo/Normal, 1: Moderado, 2: Alto/Atípico) [7, 8].
- Fundamento en Ingeniería de Características: Emula el proceso de diagnóstico médico al traducir escalas cuantitativas (del 1 al 10) en niveles cualitativos de atipia celular [7, 8]. Esta reducción de resolución actúa como un filtro de ruido que absorbe pequeñas variaciones o errores menores de medición en el laboratorio [7, 8].
- Idoneidad Algorítmica: Es la representación óptima para modelos basados en reglas e interacciones como los Árboles de Decisión [1, 2, 5]. Al evaluar divisiones (splits) sobre intervalos discretos u ordinales, el árbol construye reglas de decisión sencillas, visuales e interpretables en términos clínicos (por ejemplo: Si la uniformidad del tamaño celular es Alta, la probabilidad de malignidad se incrementa) [1, 2, 5].


#### Representación 2b: Binning por Clustering (KMeans)
- Objetivo: misma idea que R2 (3 categorías ordinales) pero con umbrales **aprendidos de los datos** en vez de la regla clínica fija.
- Método: KMeans con k=3 aplicado por variable; los clusters ordenados por centroide se mapean a 0/1/2 (Bajo/Moderado/Alto).
- Trade-off: cortes objetivos ajustados a cada distribución, pero cada variable puede tener fronteras distintas → se pierde la interpretación uniforme de la regla clínica.

In [5]:
df_rep2b = build_rep2_kmeans(df_clean)
df_rep2b.to_csv(PROCESSED_DATA_DIR / 'representacion_2b_kmeans.csv', index=False)

# Cortes aprendidos por KMeans vs. la regla clinica fija (3 / 6)
umbrales = kmeans_umbrales(df_clean)
umbrales.to_csv(REPORTS_DIR / 'binning_kmeans_umbrales.csv', index=False)
umbrales

,Clump_thickness,Uniformity_of_cell_size,Uniformity_of_cell_shape,Marginal_adhesion,Single_epithelial_cell_size,Bare_nuclei,Bland_chromatin,Normal_nucleoli,Mitoses
Corte Bajo|Moderado,4.02,3.40,3.49,3.36,3.64,3.24,4.34,3.44,3.27
Corte Moderado|Alto,7.34,7.29,7.32,7.26,7.16,7.35,7.59,7.42,7.29
Regla clinica,3 / 6,3 / 6,3 / 6,3 / 6,3 / 6,3 / 6,3 / 6,3 / 6,3 / 6


#### Representación 3: Binning + Índice de Atipia Celular Agrupado

Para preparar los datos para el **Árbol de Decisión**, se aplicaron dos pasos de ingeniería de características:

1. **Creación del Índice de Atipia Celular:**
   * Las variables `Uniformity_of_cell_size`, `Uniformity_of_cell_shape` y `Bare_nuclei` mostraron alta correlación mutua y medianas idénticas: **1.0** en tumores benignos y valores altos (**6.0**, **7.0** y **8.0**) en malignos.
   * Clínicamente las tres describen el mismo fenómeno —**atipia celular**—: células de tamaño y forma irregulares, y núcleos que se desprenden del citoplasma en células dañadas. Por eso su correlación es tan alta.
   * Para evitar redundancia y árboles innecesariamente profundos, se combinaron en una sola variable continua promediando sus valores. Con esto, el conjunto pasa de 9 a **7 variables clínicas**.

2. **Discretización por Rangos Clínicos (Binning):** las 7 variables se agrupan en las mismas 3 categorías de severidad de la Representación 2.

In [6]:
df_rep3 = build_rep3_atipia(df_clean)
df_rep3.to_csv(PROCESSED_DATA_DIR / 'representacion_3_atipia.csv', index=False)
df_rep3.head()


,Clump_thickness_bin,Marginal_adhesion_bin,Single_epithelial_cell_size_bin,Bland_chromatin_bin,Normal_nucleoli_bin,Mitoses_bin,Atipia_Celular_bin,Class
0,1,0,0,0,0,0,0,0
1,1,1,2,0,0,0,1,0
2,0,0,0,0,0,0,0,0
3,1,0,0,0,2,0,2,0
4,1,0,0,0,0,0,0,0


#### Verificación rápida: tabla cruzada de severidad vs. diagnóstico

El porcentaje de pacientes con tumor benigno que tienen valores bajos frente al
porcentaje de pacientes con tumor maligno con valores altos confirma el poder
discriminador de la atipia celular.


In [7]:
pd.crosstab(
    index=df_rep3['Atipia_Celular_bin'],
    columns=df_rep3['Class'],
    margins=True,
    normalize='columns',
)


Class,0,1,All
Atipia_Celular_bin,,,
0,0.920000,0.042017,0.468683
1,0.066667,0.310924,0.192225
2,0.013333,0.647059,0.339093


---
*Las cuatro representaciones también pueden regenerarse desde la terminal:*
`python -m wisconsin_cancer_analysis.features`